# Machine Learning Model Lifecycle

This notebook walks through the **end-to-end ML model lifecycle** in the QuantStrata library: **data** → **model** → **train** → **evaluate** → **inference**. The design is built for front-office ML quant use: generic pipelines, per-model data builders that output `tf.data.Dataset`, and standardised results for reproducibility and comparison.

## What you'll see

| Step | Component | Purpose |
|------|------------|--------|
| 1 | **Data** | `data/pricing`: build pricing data → `train_ds`, `val_ds`, `test_ds` (tf.data.Dataset) |
| 2 | **Model** | `models/pricing`: MLPPricer + config |
| 3 | **Train** | Keras `model.fit(train_ds, validation_data=val_ds)` |
| 4 | **Evaluate** | Pipeline `evaluate_model()` → standardised `EvaluationResult` |
| 5 | **Inference** | `save_model` / `load_model` / `Predictor` for deployment |

The only model-dependent parts in this flow are **data building** (`data/pricing`) and the **model** (`models/pricing`); training, evaluation, and inference are generic.

---
## 0. Configuration

Set data size, splits, training hyperparameters, and model config. In production these would be loaded from YAML/JSON or a config service.

In [ ]:
N_SAMPLES = 2000
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.7, 0.15, 0.15
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-3
SEED = 42

---
## 1. Data

Use the **per-model data builder** for pricing: `build_pricing_data()` returns `train_ds`, `val_ds`, `test_ds` as `tf.data.Dataset` and optional normalisation stats. Batching and shuffling are defined here so the pipeline sees a single interface.

In [ ]:
from src.m_learning.data.pricing import build_pricing_data

data = build_pricing_data(
    n_samples=N_SAMPLES,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    batch_size=BATCH_SIZE,
    seed=SEED,
    normalize=True,
)

print("Train batches:", len(list(data.train_ds)))
print("Metadata:", data.metadata)

---
## 2. Model

Build the **pricing model** from `models/pricing`: config-driven MLPPricer. One model class; no separate "batched" variant—batching comes from the data.

In [ ]:
import tensorflow as tf
from src.m_learning.models.pricing import create_mlp_pricer, default_pricing_config

config = default_pricing_config(
    n_features=6,
    hidden_units=[128, 64, 32],
    dropout_rate=0.1,
)
model = create_mlp_pricer(
    n_features=config.n_features,
    hidden_units=config.hidden_units,
    dropout_rate=config.dropout_rate,
    use_batch_norm=config.use_batch_norm,
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss="mse",
    metrics=["mae"],
)
model.build((None, config.n_features))
model.summary()

---
## 3. Train

Train with Keras using the datasets from step 1. The same pattern works for any model that consumes `tf.data.Dataset`.

In [ ]:
history = model.fit(
    data.train_ds,
    validation_data=data.val_ds,
    epochs=EPOCHS,
    verbose=1,
)
print("Final train loss:", history.history["loss"][-1])
print("Final val loss:", history.history["val_loss"][-1])

---
## 4. Evaluate

Use the **generic evaluation pipeline** to get a standardised `EvaluationResult` (loss, metrics, optional predictions/residuals). Same interface for any model.

In [ ]:
from src.m_learning.core.protocols import KerasTrainableAdapter
from src.m_learning.pipeline.evaluation import evaluate_model
import numpy as np

# Collect test features and targets from test_ds for pipeline API
X_test = np.vstack([x for x, _ in data.test_ds.unbatch().take(500)])
y_test = np.vstack([y for _, y in data.test_ds.unbatch().take(500)])
if y_test.ndim == 2:
    y_test = y_test.squeeze(axis=-1)

adapter = KerasTrainableAdapter(model)
eval_result = evaluate_model(
    adapter,
    X_test,
    y_test,
    metrics=["mse", "mae", "r2"],
    metadata={"split": "test", "n_samples": len(X_test)},
)
print(eval_result.summary())

---
## 5. Inference

Save model and normalisation stats, load back, and run predictions. The library provides `save_model`, `load_model`, and `Predictor` for deployment.

In [ ]:
from pathlib import Path
from src.m_learning.inference import save_model, load_model

artifact_dir = Path("/tmp/ml_lifecycle_pricer")
artifact_dir.mkdir(parents=True, exist_ok=True)

save_model(
    model,
    artifact_dir,
    feature_stats=data.feature_stats,
    target_stats=data.target_stats,
    metadata={"notebook": "ml_lifecycle"},
)
print("Saved to", artifact_dir)

artifact = load_model(artifact_dir)
preds = artifact.predict(X_test[:5], denormalize=True)
print("Predictions (first 5, denormalised):", preds)

---
## Summary

| Step | Module | Output |
|------|--------|--------|
| Data | `data.pricing.build_pricing_data` | `PricingDataResult(train_ds, val_ds, test_ds, feature_stats, target_stats)` |
| Model | `models.pricing.create_mlp_pricer` | Keras model |
| Train | `model.fit(data.train_ds, validation_data=data.val_ds)` | History |
| Evaluate | `pipeline.evaluation.evaluate_model` | `EvaluationResult` (loss, metrics, summary) |
| Inference | `pipeline.inference.save_model` / `load_model` / `predict` | Saved artifact, loaded model, predictions |

Adding a new model (e.g. GNN-RNN) follows the same lifecycle: use `data.gnn_rnn_hybrid.build_gnn_data()` and `models.gnn_rnn_hybrid`; training, evaluation, and inference pipelines stay unchanged.